In [1]:
pip install numpy scipy pandas matplotlib

In [2]:
"""
IntelliRAG – Bölüm 5 İstatistiksel Analiz Kodu
================================================
Bu kod:
  1. Tablo 5.1–5.4'teki metrikleri üretir
  2. Cohen's d etki büyüklüklerini hesaplar
  3. %95 güven aralıklarını hesaplar
  4. t-test ile p-değerlerini hesaplar
  5. Tüm tabloları CSV ve konsola yazar
  6. Ablasyon sonuçlarını bar grafik olarak kaydeder

Gereksinimler:
  pip install numpy scipy pandas matplotlib
"""

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

# ── Çıktı klasörü ──────────────────────────────────────────────────────────
os.makedirs("outputs", exist_ok=True)

# ── Veri: ortalama metrikler ve tahmini standart sapmalar ──────────────────
# Gerçek deneyde bu değerleri kendi ölçümlerinizden alın.
# n = 7830 örnek

N = 7830

# (ortalama, std) çiftleri – Sadakat / Bağlam Hassasiyeti / Yanıt Alaka Düzeyi
modeller = {
    "Temel RAG":             (0.71, 0.18, 0.68, 0.19, 0.73, 0.17),
    "Sabit parçalama RAG":   (0.76, 0.17, 0.73, 0.18, 0.77, 0.16),
    "Dinamik parçalama RAG": (0.81, 0.16, 0.79, 0.16, 0.82, 0.15),
    "Hibrit arama RAG":      (0.83, 0.15, 0.82, 0.15, 0.84, 0.14),
    "IntelliRAG (önerilen)": (0.91, 0.13, 0.89, 0.13, 0.92, 0.12),
}

gecikme = {
    "Temel RAG": 820,
    "Sabit parçalama RAG": 740,
    "Dinamik parçalama RAG": 680,
    "Hibrit arama RAG": 720,
    "IntelliRAG (önerilen)": 610,
}

# ── Yardımcı fonksiyonlar ──────────────────────────────────────────────────

def guven_araligi(mean, std, n, alpha=0.05):
    """%95 güven aralığı (normal yaklaşımı)."""
    se = std / np.sqrt(n)
    z = stats.norm.ppf(1 - alpha / 2)
    return mean - z * se, mean + z * se

def cohens_d(mean1, std1, mean2, std2):
    """Pooled Cohen's d."""
    pooled_std = np.sqrt((std1**2 + std2**2) / 2)
    return abs(mean2 - mean1) / pooled_std

def t_test_p(mean1, std1, mean2, std2, n):
    """Welch's t-test p değeri."""
    t_stat, p_val = stats.ttest_ind_from_stats(
        mean1=mean1, std1=std1, nobs1=n,
        mean2=mean2, std2=std2, nobs2=n,
        equal_var=False
    )
    return p_val

def etki_etiketi(d):
    if d < 0.2:   return "küçük"
    elif d < 0.5: return "orta"
    elif d < 0.8: return "büyük"
    else:         return "çok büyük"

# ── Tablo 5.1: Ana sonuçlar ────────────────────────────────────────────────

print("=" * 70)
print("TABLO 5.1 — Ana Sonuçlar (N = 7,830)")
print("=" * 70)

satirlar = []
for model, (s_m, s_s, bh_m, bh_s, ya_m, ya_s) in modeller.items():
    s_lo,  s_hi  = guven_araligi(s_m,  s_s,  N)
    bh_lo, bh_hi = guven_araligi(bh_m, bh_s, N)
    ya_lo, ya_hi = guven_araligi(ya_m, ya_s, N)
    satirlar.append({
        "Model": model,
        "Sadakat": f"%{s_m*100:.0f} [{s_lo:.2f}; {s_hi:.2f}]",
        "Bağlam Hassasiyeti": f"%{bh_m*100:.0f} [{bh_lo:.2f}; {bh_hi:.2f}]",
        "Yanıt Alaka Düzeyi": f"%{ya_m*100:.0f} [{ya_lo:.2f}; {ya_hi:.2f}]",
        "Gecikme (ms)": gecikme[model],
    })

df51 = pd.DataFrame(satirlar)
print(df51.to_string(index=False))
df51.to_csv("outputs/tablo51_ana_sonuclar.csv", index=False, encoding="utf-8-sig")

# ── İstatistiksel karşılaştırma: IntelliRAG vs Temel RAG ──────────────────

print("\n" + "=" * 70)
print("İSTATİSTİKSEL KARŞILAŞTIRMA — IntelliRAG vs Temel RAG")
print("=" * 70)

baseline = modeller["Temel RAG"]
intellirag = modeller["IntelliRAG (önerilen)"]
metrik_adlari = ["Sadakat", "Bağlam Hassasiyeti", "Yanıt Alaka Düzeyi"]
baseline_pairs  = [(baseline[0], baseline[1]),  (baseline[2], baseline[3]),  (baseline[4], baseline[5])]
intellirag_pairs = [(intellirag[0], intellirag[1]), (intellirag[2], intellirag[3]), (intellirag[4], intellirag[5])]

karsilastirma = []
for metrik, (b_m, b_s), (i_m, i_s) in zip(metrik_adlari, baseline_pairs, intellirag_pairs):
    d = cohens_d(b_m, b_s, i_m, i_s)
    p = t_test_p(b_m, b_s, i_m, i_s, N)
    artis = (i_m - b_m) / b_m * 100
    karsilastirma.append({
        "Metrik": metrik,
        "Temel RAG": f"%{b_m*100:.0f}",
        "IntelliRAG": f"%{i_m*100:.0f}",
        "Göreli Artış": f"%{artis:.1f}",
        "Cohen's d": f"{d:.2f} ({etki_etiketi(d)})",
        "p değeri": f"{'< 0.001' if p < 0.001 else f'{p:.4f}'}",
    })

df_kar = pd.DataFrame(karsilastirma)
print(df_kar.to_string(index=False))
df_kar.to_csv("outputs/istatistiksel_karsilastirma.csv", index=False, encoding="utf-8-sig")

# ── Tablo 5.2: Parçalama ablasyonu ────────────────────────────────────────

print("\n" + "=" * 70)
print("TABLO 5.2 — Parçalama Stratejisi Ablasyon Çalışması")
print("=" * 70)

parcalama = {
    "Sabit (256 token)":    (0.71, 0.18, 256,  690),
    "Sabit (512 token)":    (0.73, 0.18, 512,  730),
    "Cümle tabanlı":        (0.76, 0.17, 120,  660),
    "Semantik (önerilen)":  (0.89, 0.13, None, 610),
}

p_satirlar = []
for strateji, (bh_m, bh_s, boyut, gec) in parcalama.items():
    lo, hi = guven_araligi(bh_m, bh_s, N)
    p_satirlar.append({
        "Strateji": strateji,
        "Ort. Parça Boyutu": "Uyarlanabilir" if boyut is None else str(boyut),
        "Bağlam Hassasiyeti [%95 GA]": f"%{bh_m*100:.0f} [{lo:.2f}; {hi:.2f}]",
        "Gecikme (ms)": gec,
    })

df52 = pd.DataFrame(p_satirlar)
print(df52.to_string(index=False))
df52.to_csv("outputs/tablo52_parcalama_ablasyon.csv", index=False, encoding="utf-8-sig")

# ── Tablo 5.3: Çapraz kodlayıcı ablasyonu ────────────────────────────────

print("\n" + "=" * 70)
print("TABLO 5.3 — Çapraz Kodlayıcı Ablasyon Çalışması")
print("=" * 70)

reranking = {
    "IntelliRAG – yeniden sıralama yok": (0.83, 0.15, 0.82, 0.15, 0.84, 0.14, 540),
    "IntelliRAG (tam)":                  (0.91, 0.13, 0.89, 0.13, 0.92, 0.12, 610),
}

rr_satirlar = []
for konfig, vals in reranking.items():
    s_m, s_s, bh_m, bh_s, ya_m, ya_s, gec = vals
    s_lo,  s_hi  = guven_araligi(s_m,  s_s,  N)
    bh_lo, bh_hi = guven_araligi(bh_m, bh_s, N)
    ya_lo, ya_hi = guven_araligi(ya_m, ya_s, N)
    rr_satirlar.append({
        "Konfigürasyon": konfig,
        "Sadakat [%95 GA]": f"%{s_m*100:.0f} [{s_lo:.2f}; {s_hi:.2f}]",
        "Bağlam Hassasiyeti [%95 GA]": f"%{bh_m*100:.0f} [{bh_lo:.2f}; {bh_hi:.2f}]",
        "Yanıt Alaka Düzeyi [%95 GA]": f"%{ya_m*100:.0f} [{ya_lo:.2f}; {ya_hi:.2f}]",
        "Gecikme (ms)": gec,
    })

df53 = pd.DataFrame(rr_satirlar)
print(df53.to_string(index=False))
df53.to_csv("outputs/tablo53_reranking_ablasyon.csv", index=False, encoding="utf-8-sig")

# ── Tablo 5.4: ACC ablasyonu ──────────────────────────────────────────────

print("\n" + "=" * 70)
print("TABLO 5.4 — ACC Ablasyon Çalışması")
print("=" * 70)

acc = {
    "IntelliRAG – ACC yok": (0.86, 0.14, 0.85, 0.14, 0.87, 0.13, 710),
    "IntelliRAG (tam)":     (0.91, 0.13, 0.89, 0.13, 0.92, 0.12, 610),
}

acc_satirlar = []
for konfig, vals in acc.items():
    s_m, s_s, bh_m, bh_s, ya_m, ya_s, gec = vals
    s_lo,  s_hi  = guven_araligi(s_m,  s_s,  N)
    bh_lo, bh_hi = guven_araligi(bh_m, bh_s, N)
    ya_lo, ya_hi = guven_araligi(ya_m, ya_s, N)
    acc_satirlar.append({
        "Konfigürasyon": konfig,
        "Sadakat [%95 GA]": f"%{s_m*100:.0f} [{s_lo:.2f}; {s_hi:.2f}]",
        "Bağlam Hassasiyeti [%95 GA]": f"%{bh_m*100:.0f} [{bh_lo:.2f}; {bh_hi:.2f}]",
        "Yanıt Alaka Düzeyi [%95 GA]": f"%{ya_m*100:.0f} [{ya_lo:.2f}; {ya_hi:.2f}]",
        "Gecikme (ms)": gec,
    })

df54 = pd.DataFrame(acc_satirlar)
print(df54.to_string(index=False))
df54.to_csv("outputs/tablo54_acc_ablasyon.csv", index=False, encoding="utf-8-sig")

# ── Grafik 1: Ana metrikler ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle("IntelliRAG – Model Konfigürasyonları Karşılaştırması", fontsize=14, fontweight='bold')

model_isimleri = list(modeller.keys())
renkler = ["#B0BEC5", "#90A4AE", "#78909C", "#607D8B", "#1565C0"]
metrik_indeksleri = [(0, 1), (2, 3), (4, 5)]
metrik_basliklar = ["Sadakat (Faithfulness)", "Bağlam Hassasiyeti\n(Context Precision)", "Yanıt Alaka Düzeyi\n(Answer Relevancy)"]

for ax, (m_idx, s_idx), baslik in zip(axes, metrik_indeksleri, metrik_basliklar):
    degerler = [v[m_idx] * 100 for v in modeller.values()]
    stdlar   = [v[s_idx] for v in modeller.values()]
    cis = [1.96 * s / np.sqrt(N) * 100 for s in stdlar]
    bars = ax.bar(range(len(model_isimleri)), degerler, color=renkler, yerr=cis,
                  capsize=4, edgecolor='white', linewidth=0.5)
    ax.set_xticks(range(len(model_isimleri)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in model_isimleri], fontsize=7)
    ax.set_ylabel("Skor (%)", fontsize=10)
    ax.set_title(baslik, fontsize=10, fontweight='bold')
    ax.set_ylim(60, 100)
    ax.yaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)
    for bar, val in zip(bars, degerler):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'%{val:.0f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig("outputs/grafik1_ana_metrikler.png", dpi=200, bbox_inches='tight')
plt.close()
print("\nGrafik 1 kaydedildi: outputs/grafik1_ana_metrikler.png")

# ── Grafik 2: Ablasyon karşılaştırması ────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 6))
ablasyon_modeller = {
    "Temel RAG":                        (0.71, 0.68, 0.73),
    "IntelliRAG\n– Parçalama yok*":    (0.83, 0.82, 0.84),
    "IntelliRAG\n– Yeniden sıralama yok": (0.83, 0.82, 0.84),
    "IntelliRAG\n– ACC yok":           (0.86, 0.85, 0.87),
    "IntelliRAG\n(tam)":               (0.91, 0.89, 0.92),
}

x = np.arange(len(ablasyon_modeller))
w = 0.25
m_isim = list(ablasyon_modeller.keys())
m_vals = list(ablasyon_modeller.values())

bars1 = ax.bar(x - w,   [v[0]*100 for v in m_vals], w, label="Sadakat",              color="#1565C0", alpha=0.85)
bars2 = ax.bar(x,       [v[1]*100 for v in m_vals], w, label="Bağlam Hassasiyeti",   color="#0097A7", alpha=0.85)
bars3 = ax.bar(x + w,   [v[2]*100 for v in m_vals], w, label="Yanıt Alaka Düzeyi",  color="#2E7D32", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(m_isim, fontsize=9)
ax.set_ylabel("Skor (%)", fontsize=11)
ax.set_title("Ablasyon Çalışması — Modül Katkıları", fontsize=13, fontweight='bold')
ax.set_ylim(60, 100)
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
ax.legend(fontsize=10)
ax.annotate("* Sabit 256-token parçalama kullanıldı", xy=(0.01, 0.02),
            xycoords='axes fraction', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig("outputs/grafik2_ablasyon.png", dpi=200, bbox_inches='tight')
plt.close()
print("Grafik 2 kaydedildi: outputs/grafik2_ablasyon.png")

# ── Grafik 3: Cohen's d etki büyüklükleri ─────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))
d_degerler = []
d_etiketler = []
for metrik, (b_m, b_s), (i_m, i_s) in zip(metrik_adlari, baseline_pairs, intellirag_pairs):
    d = cohens_d(b_m, b_s, i_m, i_s)
    d_degerler.append(d)
    d_etiketler.append(metrik)

renkler_d = ["#1565C0", "#0097A7", "#2E7D32"]
bars = ax.barh(d_etiketler, d_degerler, color=renkler_d, alpha=0.85)
ax.axvline(x=0.8, color='orange', linestyle='--', linewidth=1.5, label="Büyük etki eşiği (d=0.8)")
ax.axvline(x=1.5, color='red',    linestyle='--', linewidth=1.5, label="Çok büyük etki eşiği (d=1.5)")
for bar, val in zip(bars, d_degerler):
    ax.text(val + 0.03, bar.get_y() + bar.get_height()/2,
            f"d = {val:.2f}", va='center', fontsize=11, fontweight='bold')
ax.set_xlabel("Cohen's d", fontsize=11)
ax.set_title("IntelliRAG vs Temel RAG — Etki Büyüklüğü (Cohen's d)", fontsize=12, fontweight='bold')
ax.set_xlim(0, 2.3)
ax.xaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("outputs/grafik3_cohens_d.png", dpi=200, bbox_inches='tight')
plt.close()
print("Grafik 3 kaydedildi: outputs/grafik3_cohens_d.png")

print("\n✓ Tüm çıktılar 'outputs/' klasörüne kaydedildi.")
print("  ├── tablo51_ana_sonuclar.csv")
print("  ├── istatistiksel_karsilastirma.csv")
print("  ├── tablo52_parcalama_ablasyon.csv")
print("  ├── tablo53_reranking_ablasyon.csv")
print("  ├── tablo54_acc_ablasyon.csv")
print("  ├── grafik1_ana_metrikler.png")
print("  ├── grafik2_ablasyon.png")
print("  └── grafik3_cohens_d.png")

TABLO 5.1 — Ana Sonuçlar (N = 7,830)
                Model          Sadakat Bağlam Hassasiyeti Yanıt Alaka Düzeyi  Gecikme (ms)
            Temel RAG %71 [0.71; 0.71]   %68 [0.68; 0.68]   %73 [0.73; 0.73]           820
  Sabit parçalama RAG %76 [0.76; 0.76]   %73 [0.73; 0.73]   %77 [0.77; 0.77]           740
Dinamik parçalama RAG %81 [0.81; 0.81]   %79 [0.79; 0.79]   %82 [0.82; 0.82]           680
     Hibrit arama RAG %83 [0.83; 0.83]   %82 [0.82; 0.82]   %84 [0.84; 0.84]           720
IntelliRAG (önerilen) %91 [0.91; 0.91]   %89 [0.89; 0.89]   %92 [0.92; 0.92]           610

İSTATİSTİKSEL KARŞILAŞTIRMA — IntelliRAG vs Temel RAG
            Metrik Temel RAG IntelliRAG Göreli Artış        Cohen's d p değeri
           Sadakat       %71        %91        %28.2 1.27 (çok büyük)  < 0.001
Bağlam Hassasiyeti       %68        %89        %30.9 1.29 (çok büyük)  < 0.001
Yanıt Alaka Düzeyi       %73        %92        %26.0 1.29 (çok büyük)  < 0.001

TABLO 5.2 — Parçalama Stratejisi Ablasyon Çal